# Phase 07 — Comprehensive Evaluation & Explainable AI (XAI)

**CS114 Final Project: Vietnamese Elderly Scam Detection**  
**Tác giả:** Đặng Chí Thanh (MSSV: 25730067 — PM & Evaluation Lead)  
**Ngày hoàn thành:** 2026-09-19 (Sprint 3 Fast-Track)  

---

## 1. Mục tiêu Nghiên cứu & Thực nghiệm

- So sánh toàn diện 4 mô hình đã huấn luyện (**Naive Bayes**, **Random Forest**, **XGBoost**, **SVM Linear**) trên tập holdout test độc lập (439 mẫu — 355 Normal / 84 Scam).
- Đánh giá 5 chỉ số cốt lõi: Accuracy, Precision, Recall, F1-Score, ROC-AUC và phân tích ma trận nhầm lẫn (FN vs FP).
- Xuất các biểu đồ phân tích trực quan chuẩn xuất bản 300 DPI:
  1. Ma trận nhầm lẫn lưới 2×2 (`confusion_matrices_all.png`).
  2. Đường cong ROC lồng ghép cả 4 mô hình (`roc_curves_comparison.png`).
  3. Đường cong Precision-Recall (`precision_recall_curves.png`).
  4. Top-20 từ khóa kích hoạt bẫy tâm lý theo trọng số mô hình (`top20_keywords_barplot.png`).
  5. Phân tích đóng góp đặc trưng SHAP TreeExplainer (`shap_summary.png`).
  6. Tổng hợp kiểm soát quá khớp qua Learning Curves (`learning_curves_summary.png`).
- Khóa mô hình tối ưu (**Best Model**) bàn giao cho Demo UI (Phase 08) và Chương 4 Báo cáo (Phase 09).

## 2. Setup & Nạp Thư Viện

In [ ]:
import sys
import json
from pathlib import Path
import numpy as np
import pandas as pd
import joblib
import matplotlib.pyplot as plt
import seaborn as sns

PROJECT_ROOT = Path('.').resolve()
if not (PROJECT_ROOT / 'src').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.utils.helpers import RANDOM_STATE
from src.evaluation.metrics_calculator import evaluate_all_models, format_markdown_table
from src.evaluation.plot_comparator import (
    plot_confusion_matrices_grid, plot_roc_comparison,
    plot_precision_recall_comparison, plot_top20_keywords_comparison,
    plot_learning_curves_summary,
)
from src.evaluation.xai_explainer import (
    extract_consensus_keywords, extract_keywords_from_model,
    compute_and_save_shap_plot,
)

DATA_DIR = PROJECT_ROOT / 'data' / 'processed'
MODELS_DIR = PROJECT_ROOT / 'models'
FIGURES_DIR = PROJECT_ROOT / 'reports' / 'figures'
FIGURES_DIR.mkdir(parents=True, exist_ok=True)
print(f'[✓] Setup complete. RANDOM_STATE={RANDOM_STATE}')

## 3. Nạp Dữ Liệu Test & Mô Hình Huấn Luyện

In [ ]:
X_test = joblib.load(DATA_DIR / 'X_test.pkl')
y_test = joblib.load(DATA_DIR / 'y_test.pkl')
with open(DATA_DIR / 'feature_names.json', 'r', encoding='utf-8') as f:
    feature_names = json.load(f)

print(f'Test matrix shape: {X_test.shape} (439 mẫu test, 5015 đặc trưng)')
print(f'Tỷ lệ nhãn: Bình thường = {(y_test == 0).sum()} mẫu, Lừa đảo = {(y_test == 1).sum()} mẫu')

models = {
    'XGBoost': joblib.load(MODELS_DIR / 'xgboost.pkl'),
    'SVM Linear': joblib.load(MODELS_DIR / 'svm_linear.pkl'),
    'Random Forest': joblib.load(MODELS_DIR / 'random_forest.pkl'),
    'Naive Bayes': joblib.load(MODELS_DIR / 'naive_bayes.pkl'),
}
print(f'[✓] Đã nạp thành công 4 mô hình: {list(models.keys())}')

## 4. Tính Toán & Lập Bảng So Sánh Hiệu Năng

In [ ]:
metrics_df = evaluate_all_models(models, X_test, y_test)
print('=== BẢNG SO SÁNH HIỆU NĂNG TỔNG HỢP (TEST SET) ===')
print(format_markdown_table(metrics_df))

# Cập nhật lưu trữ vào reports/model_comparison.csv
csv_path = PROJECT_ROOT / 'reports' / 'model_comparison.csv'
records = []
meta = {
    'XGBoost': ('5015 (TF-IDF + 15 aux)', 'n_estimators=300, max_depth=5, lr=0.05, subsample=0.7, scale_pos_weight=4.22', 'Passed (Phase 05 — Cam)'),
    'SVM Linear': ('5015 (TF-IDF + 15 aux)', 'C=10.0 (CalibratedClassifierCV 5-fold)', 'Passed (Phase 06 — Cam)'),
    'Random Forest': ('5015 (TF-IDF + 15 aux)', 'n_estimators=200, max_depth=50, split=2, leaf=2, max_feat=0.2', 'Passed (Phase 05 — Son)'),
    'Naive Bayes': ('5000 (TF-IDF non-negative)', 'alpha=0.01 (GridSearchCV 5-fold)', 'Passed (Phase 04 — Cam)'),
}
for _, row in metrics_df.iterrows():
    m_name = row['model']
    feat, hyper, st = meta.get(m_name, ('5015', 'Tuned', 'Passed'))
    records.append({
        'Model': m_name,
        'Features': feat,
        'Hyperparameters': hyper,
        'Accuracy': round(row['accuracy'], 4),
        'Precision': round(row['precision'], 4),
        'Recall': round(row['recall'], 4),
        'F1-Score': round(row['f1'], 4),
        'ROC-AUC': round(row['roc_auc'], 4),
        'Status': st,
    })
pd.DataFrame(records).to_csv(csv_path, index=False)
print(f'[✓] Đã đồng bộ model_comparison.csv tại: {csv_path}')

## 5. Trực Quan Hóa Ma Trận Nhầm Lẫn (Confusion Matrix Grid 2×2)

In [ ]:
p_cm = plot_confusion_matrices_grid(models, X_test, y_test, FIGURES_DIR)
print(f'[✓] Đã xuất biểu đồ: {p_cm}')

# Hiển thị trực tiếp trong notebook
img_cm = plt.imread(str(p_cm))
plt.figure(figsize=(10, 8))
plt.imshow(img_cm)
plt.axis('off')
plt.show()

## 6. So Sánh Đường Cong ROC & Precision-Recall

In [ ]:
p_roc = plot_roc_comparison(models, X_test, y_test, FIGURES_DIR)
p_pr = plot_precision_recall_comparison(models, X_test, y_test, FIGURES_DIR)
print(f'[✓] Đã xuất ROC: {p_roc}')
print(f'[✓] Đã xuất PR: {p_pr}')

fig, axes = plt.subplots(1, 2, figsize=(16, 7))
axes[0].imshow(plt.imread(str(p_roc)))
axes[0].axis('off')
axes[1].imshow(plt.imread(str(p_pr)))
axes[1].axis('off')
plt.tight_layout()
plt.show()

## 7. Tổng Hợp Phân Tích Quá Khớp (Learning Curves)

In [ ]:
p_lc = plot_learning_curves_summary(FIGURES_DIR)
print(f'[✓] Đã xuất tổng hợp Learning Curves: {p_lc}')
plt.figure(figsize=(12, 9))
plt.imshow(plt.imread(str(p_lc)))
plt.axis('off')
plt.show()

## 8. Trích Xuất Từ Khóa Cảnh Báo Lừa Đảo (Consensus XAI)

In [ ]:
consensus_kws = extract_consensus_keywords(models, feature_names, top_n=20)
p_kws = plot_top20_keywords_comparison(consensus_kws, FIGURES_DIR)
print(f'[✓] Đã xuất biểu đồ Top-20 từ khóa cảnh báo: {p_kws}')

print('=== TOP 20 TỪ KHÓA BẪY TÂM LÝ LỪA ĐẢO ĐỒNG THUẬN CAO NHẤT ===')
for rank, (word, score) in enumerate(consensus_kws, start=1):
    print(f'{rank:02d}. {word:<25} (Trọng số chuẩn hóa: {score:.4f})')

## 9. Phân Tích Đóng Góp Giá Trị Đặc Trưng SHAP (TreeExplainer)

In [ ]:
sample_indices = list(range(min(150, X_test.shape[0])))
X_sample = X_test[sample_indices]
p_shap = compute_and_save_shap_plot(models['XGBoost'], X_sample, feature_names, FIGURES_DIR)
print(f'[✓] Đã xuất biểu đồ SHAP Beeswarm: {p_shap}')

plt.figure(figsize=(10, 8))
plt.imshow(plt.imread(str(p_shap)))
plt.axis('off')
plt.show()

## 10. Kết Luận & Chọn Mô Hình Tối Ưu (Best Model Selection)

1. **Mô hình Xuất sắc Nhất (Best Overall Model):** **XGBoost Classifier**  
   - **F1-Score:** `0.9535` (Cao nhất trong cả 4 mô hình).  
   - **Recall Lớp Lừa Đảo:** `97.62%` (Chỉ bỏ sót đúng 2/84 mẫu lừa đảo trong tập kiểm thử).  
   - **Precision:** `93.18%` (Chỉ báo nhầm 6/355 mẫu bình thường).  
   - **ROC-AUC:** `0.9984`.  
2. **Mô hình Tuyến tính Tối ưu Nhất (Best Linear Model):** **Linear SVM (Calibrated)**  
   - **ROC-AUC:** `0.9986` (Khả năng phân tách xác suất tốt nhất).  
   - **F1-Score:** `0.9474`, Recall: `96.43%`.  
3. **Quyết định Triển Khai (Deployment Decision):**  
   - Cả 4 mô hình đều vượt xa chỉ tiêu đặt ra tại `BRAINSTORM.md` (`F1 ≥ 88%`, `Recall ≥ 92%`).  
   - Gói suy luận `best_model_bundle.pkl` được đóng gói bằng XGBoost / Linear SVM phục vụ Phase 08 (Demo UI).